# Paclitaxel Multi-omics Classification

This notebook evaluates fold-safe supervised classifiers for paclitaxel response prediction using the cleaned RNA, CNV, and mutation matrices from `1-data_preprocessing.ipynb`.

Primary endpoint: a fixed threshold on the batch-level growth-curve `slope.treatment`, matching case study 3's own convention exactly (same response quantity, same fixed cutoff) rather than a distribution-derived split.

- **Resistant**: `slope.treatment` >= 0 deg (net growth)
- **Sensitive**: `slope.treatment` < 0 deg (net tumor regression)

mRECIST is kept in the metadata as a reference/QC column only -- it is not used to define the modeling endpoint. `angle` is also kept as a reference/QC column (it was the original candidate endpoint; see 1-data_preprocessing.ipynb's "Response distribution" diagnostic for why a median or gap-based cutoff on either quantity was set aside in favor of this fixed one).

Feature selection is performed inside each cross-validation fold. External gene priors, including the curated paclitaxel gene list and the NEST-VNN gene list, are fixed a priori and force-retained when present.

## Imports and Configuration

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from pipeline_utils import make_case4_paths

paths = make_case4_paths()
DATA_DIR = paths["proc"]
RESULTS_DIR = paths["results"]

sys.path.append(str(paths["project"] / "4_paclitaxel"))
from feature_sets import (
    add_binary_response,
    load_pax_genes,
    load_nest_genes,
    split_x_class_y,
    select_features_by_omics,
)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
POS_LABEL = 1  # Sensitive

print("Project root:", paths["project"])
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Project root: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study
Data dir: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/4_paclitaxel
Results dir: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/results/4_paclitaxel


## Load Data and Define Binary Response

In [2]:
metadata_df = pd.read_csv(DATA_DIR / "metadata.csv", index_col=0)

# Fixed slope.treatment < 0 cutoff (matching case 3) -- a constant, not derived
# from any particular subset, so it can be applied identically to every omics
# subset below (rna_all, rna_paired/cnv/mutation) without needing to thread a
# shared value through every call site the way a median-based threshold would.
metadata_df = add_binary_response(metadata_df)
rna_df = add_binary_response(pd.read_csv(DATA_DIR / "rna_logtpm.csv", index_col=0))
cnv_df = add_binary_response(pd.read_csv(DATA_DIR / "cnv.csv", index_col=0))
mutation_df = add_binary_response(pd.read_csv(DATA_DIR / "mutation.csv", index_col=0))

pax_genes = load_pax_genes()
nest_genes = load_nest_genes()

all3_ids = sorted(set(rna_df.index) & set(cnv_df.index) & set(mutation_df.index))
rna_paired_df = rna_df.loc[all3_ids]
cnv_paired_df = cnv_df.loc[all3_ids]
mutation_paired_df = mutation_df.loc[all3_ids]

for name, df in [
    ("metadata", metadata_df),
    ("rna_all", rna_df),
    ("rna_paired", rna_paired_df),
    ("cnv", cnv_paired_df),
    ("mutation", mutation_paired_df),
]:
    print(f"\n{name} {df.shape}")
    print(df["mRECIST"].value_counts().to_string())
    print(df["response_binary"].value_counts().to_string())

print(f"\nPaclitaxel prior genes: {len(pax_genes)}")
print("NEST prior genes:", len(nest_genes))



metadata (80, 5)
mRECIST
SD    33
PD    19
PR    14
CR    14
response_binary
Sensitive    47
Resistant    33

rna_all (67, 49353)
mRECIST
SD    28
PD    15
CR    13
PR    11
response_binary
Sensitive    39
Resistant    28

rna_paired (62, 49353)
mRECIST
SD    26
PD    14
CR    12
PR    10
response_binary
Sensitive    36
Resistant    26

cnv (62, 10318)
mRECIST
SD    26
PD    14
CR    12
PR    10
response_binary
Sensitive    36
Resistant    26

mutation (62, 13034)
mRECIST
SD    26
PD    14
CR    12
PR    10
response_binary
Sensitive    36
Resistant    26

Paclitaxel prior genes: 69
NEST prior genes: 718


## Response QC

How does the fixed slope.treatment<0 `response_binary` (the actual modeling endpoint) relate to the original mRECIST clinical categories, kept in the metadata as a reference column only?

In [3]:
MRECIST_ORDER_QC = ["PD", "SD", "PR", "CR"]

concordance = pd.crosstab(metadata_df["mRECIST"], metadata_df["response_binary"]).reindex(MRECIST_ORDER_QC)
print("mRECIST category x fixed slope.treatment<0 response_binary:")
print(concordance)

# mRECIST-collapsed reference: PD/SD -> Resistant, PR/CR -> Sensitive. This is
# not a ground truth -- slope.treatment was chosen over mRECIST deliberately
# (see intro markdown) -- it just documents where the two definitions agree/disagree.
mrecist_collapsed_map = {"PD": "Resistant", "SD": "Resistant", "PR": "Sensitive", "CR": "Sensitive"}
mrecist_collapsed = metadata_df["mRECIST"].map(mrecist_collapsed_map)
agreement = (metadata_df["response_binary"] == mrecist_collapsed).mean()
print(f"\nAgreement with mRECIST-collapsed (PD/SD=Resistant, PR/CR=Sensitive): {agreement:.1%}")

flipped = metadata_df.loc[
    metadata_df["response_binary"] != mrecist_collapsed, ["mRECIST", "slope.treatment", "angle", "response_binary"]
].sort_values("slope.treatment")
print(f"\n{len(flipped)} models where the fixed slope.treatment<0 label disagrees with mRECIST-collapsed:")

response_qc_df = flipped.reset_index()
response_qc_df.to_csv(RESULTS_DIR / "response_metric_qc.csv", index=False)
response_qc_df


mRECIST category x fixed slope.treatment<0 response_binary:
response_binary  Resistant  Sensitive
mRECIST                              
PD                      17          2
SD                      16         17
PR                       0         14
CR                       0         14

Agreement with mRECIST-collapsed (PD/SD=Resistant, PR/CR=Sensitive): 76.2%

19 models where the fixed slope.treatment<0 label disagrees with mRECIST-collapsed:


,model.id,mRECIST,slope.treatment,angle,response_binary
0,S66684,SD,-74.658943,150.195285,Sensitive
1,S65465,SD,-71.203209,156.885905,Sensitive
2,REF041,SD,-68.656885,138.655568,Sensitive
3,S85277,PD,-60.929419,145.641352,Sensitive
4,REF025,SD,-57.560135,133.930367,Sensitive
5,S66817,SD,-56.150139,126.983285,Sensitive
6,S17518,SD,-54.589820,142.880996,Sensitive
7,DCPTOB19,SD,-53.243205,117.888330,Sensitive
8,S48041,SD,-52.390662,123.335970,Sensitive
9,S70620,SD,-50.154840,133.267358,Sensitive


## Classifier and Metric Helpers

In [4]:
def classification_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "auroc": roc_auc_score(y_true, y_score),
        "auprc": average_precision_score(y_true, y_score),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def make_classification_models(random_state=RANDOM_STATE):
    # Simplified to two models: random_forest was the consistently best or
    # near-best performer across every omics/analysis combination, logistic_l2
    # the consistently strong runner-up; logistic_l1 and svm_rbf were dropped
    # after ranking all four models within each (analysis, omics) group across
    # three independent runs (uncapped angle, windowed angle, and a
    # slope.treatment comparison run) -- svm_rbf and logistic_l1 were reliably
    # the bottom two, especially in the RNA-driven groups central to the
    # paper's story (mean rank 3.0-4.0 vs. 1.0-2.0 for random_forest/logistic_l2
    # in all three runs).
    return {
        "logistic_l2": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l2",
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                )),
            ]),
            {"model__C": [0.01, 0.1, 1.0, 10.0]},
        ),
        "random_forest": (
            RandomForestClassifier(
                n_estimators=500,
                max_features="sqrt",
                class_weight="balanced",
                random_state=random_state,
                n_jobs=-1,
            ),
            {
                "max_depth": [None, 3, 5],
                "min_samples_leaf": [2, 3, 5],
            },
        ),
    }


def tune_model(model, param_grid, X_train, y_train):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_

## Single-omics Evaluation

In [5]:
def evaluate_single_omics(name, df, prior_mode="pax"):
    X, y = split_x_class_y(df)
    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    models = make_classification_models()

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        selected = select_features_by_omics(name, X_train_raw, prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        X_train = X_train_raw[selected]
        X_test = X_test_raw[selected]

        for model_name, (model, grid) in models.items():
            best_model, best_params = tune_model(clone(model), grid, X_train, y_train)
            y_pred = best_model.predict(X_test)
            y_score = positive_class_score(best_model, X_test)
            metrics = classification_metrics(y_test, y_pred, y_score)

            rows.append({
                "analysis": "single_omics",
                "omics": name,
                "prior_mode": prior_mode,
                "model": model_name,
                "fold": fold_id,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_features": len(selected),
                "n_pax_genes": len(set(selected) & pax_genes),
                "n_nest_genes": len(set(selected) & nest_genes),
                "best_params": str(best_params),
                **metrics,
            })

            predictions.extend([
                {
                    "analysis": "single_omics",
                    "omics": name,
                    "prior_mode": prior_mode,
                    "model": model_name,
                    "fold": fold_id,
                    "sample_id": sample_id,
                    "y_true": int(true),
                    "y_true_label": "Sensitive" if true == 1 else "Resistant",
                    "y_pred": int(pred),
                    "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                    "y_score_sensitive": score,
                }
                for sample_id, true, pred, score in zip(X_test.index, y_test, y_pred, y_score)
            ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


## Run Single-omics Baselines

This evaluates RNA on all available RNA samples, and RNA/CNV/mutation on the 62-sample paired cohort for fair comparison.

In [6]:
single_results = []
single_predictions = []

single_jobs = [
    ("rna_all", rna_df),
    ("rna_paired", rna_paired_df),
    ("cnv", cnv_paired_df),
    ("mutation", mutation_paired_df),
]

for name, df in single_jobs:
    print(f"Running {name}...")
    result_df, pred_df = evaluate_single_omics(name, df, prior_mode="pax")
    single_results.append(result_df)
    single_predictions.append(pred_df)

single_results_df = pd.concat(single_results, ignore_index=True)
single_predictions_df = pd.concat(single_predictions, ignore_index=True)

single_results_df.to_csv(RESULTS_DIR / "single_omics_classification_cv_results.csv", index=False)
single_predictions_df.to_csv(RESULTS_DIR / "single_omics_classification_cv_predictions.csv", index=False)

single_results_df.groupby(["omics", "model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running rna_all...


Running rna_paired...


Running cnv...


Running mutation...


auroc  auprc  balanced_accuracy     f1  sensitivity  \
omics      model                                                                
cnv        logistic_l2    0.510  0.666              0.498  0.556        0.546   
           random_forest  0.527  0.684              0.508  0.627        0.691   
mutation   logistic_l2    0.374  0.554              0.453  0.512        0.500   
           random_forest  0.418  0.578              0.459  0.627        0.734   
rna_all    logistic_l2    0.618  0.725              0.588  0.620        0.600   
           random_forest  0.657  0.743              0.597  0.733        0.856   
rna_paired logistic_l2    0.671  0.758              0.620  0.667        0.658   
           random_forest  0.716  0.779              0.607  0.744        0.877   

                          specificity  n_features  
omics      model                                   
cnv        logistic_l2          0.451      492.12  
           random_forest        0.325      492.12  
mutation   logistic_l2          0.405      427.56  
           random_forest        0.184      427.56  
rna_all    logistic_l2          0.576     2055.00  
           random_forest        0.337     2055.00  
rna_paired logistic_l2          0.581     2055.68  
           random_forest        0.337     2055.68

## Early-fusion Multi-omics Evaluation

For each fold, feature selection is performed independently within RNA, CNV, and mutation training folds. The selected matrices are concatenated and evaluated as a single feature table.

In [7]:
def evaluate_early_fusion(prior_mode="pax"):
    X_rna, y = split_x_class_y(rna_paired_df)
    X_cnv, _ = split_x_class_y(cnv_paired_df)
    X_mut, _ = split_x_class_y(mutation_paired_df)

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    models = make_classification_models()

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_rna, y), start=1):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        selected_rna = select_features_by_omics("rna_paired", X_rna.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        selected_cnv = select_features_by_omics("cnv", X_cnv.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        selected_mut = select_features_by_omics("mutation", X_mut.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)

        X_train = pd.concat(
            [
                X_rna.iloc[train_idx][selected_rna].add_prefix("rna__"),
                X_cnv.iloc[train_idx][selected_cnv].add_prefix("cnv__"),
                X_mut.iloc[train_idx][selected_mut].add_prefix("mut__"),
            ],
            axis=1,
        )
        X_test = pd.concat(
            [
                X_rna.iloc[test_idx][selected_rna].add_prefix("rna__"),
                X_cnv.iloc[test_idx][selected_cnv].add_prefix("cnv__"),
                X_mut.iloc[test_idx][selected_mut].add_prefix("mut__"),
            ],
            axis=1,
        )

        for model_name, (model, grid) in models.items():
            best_model, best_params = tune_model(clone(model), grid, X_train, y_train)
            y_pred = best_model.predict(X_test)
            y_score = positive_class_score(best_model, X_test)
            metrics = classification_metrics(y_test, y_pred, y_score)

            rows.append({
                "analysis": "early_fusion",
                "omics": "rna_cnv_mutation",
                "prior_mode": prior_mode,
                "model": model_name,
                "fold": fold_id,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_features": X_train.shape[1],
                "n_rna_features": len(selected_rna),
                "n_cnv_features": len(selected_cnv),
                "n_mutation_features": len(selected_mut),
                "best_params": str(best_params),
                **metrics,
            })

            predictions.extend([
                {
                    "analysis": "early_fusion",
                    "omics": "rna_cnv_mutation",
                    "prior_mode": prior_mode,
                    "model": model_name,
                    "fold": fold_id,
                    "sample_id": sample_id,
                    "y_true": int(true),
                    "y_true_label": "Sensitive" if true == 1 else "Resistant",
                    "y_pred": int(pred),
                    "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                    "y_score_sensitive": score,
                }
                for sample_id, true, pred, score in zip(X_test.index, y_test, y_pred, y_score)
            ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [8]:
print("Running early fusion...")
early_results_df, early_predictions_df = evaluate_early_fusion(prior_mode="pax")

early_results_df.to_csv(RESULTS_DIR / "early_fusion_classification_cv_results.csv", index=False)
early_predictions_df.to_csv(RESULTS_DIR / "early_fusion_classification_cv_predictions.csv", index=False)

early_results_df.groupby(["model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running early fusion...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
logistic_l2,0.588,0.696,0.610,0.658,0.659,0.560,2975.36
random_forest,0.699,0.777,0.601,0.734,0.861,0.341,2975.36


## Late-fusion / Stacking Multi-omics Evaluation

Late fusion trains one base classifier per omics layer and combines their out-of-fold probability predictions with a simple meta-classifier. This tests whether CNV and mutation add complementary information without concatenating thousands of features directly with RNA.

For runtime, the default stack uses `logistic_l2` base models and a ridge-like logistic meta-model. The implementation keeps feature selection inside the relevant training folds.

In [9]:
def get_model_spec(model_name):
    models = make_classification_models()
    if model_name not in models:
        raise ValueError(f"Unknown model_name: {model_name}")
    return models[model_name]


def fit_predict_one_omics(omics_name, X_train_raw, y_train, X_test_raw, model_name, prior_mode):
    selected = select_features_by_omics(
        omics_name,
        X_train_raw,
        prior_mode=prior_mode,
        pax_genes=pax_genes,
        nest_genes=nest_genes,
    )
    model, grid = get_model_spec(model_name)
    best_model, best_params = tune_model(clone(model), grid, X_train_raw[selected], y_train)
    test_score = positive_class_score(best_model, X_test_raw[selected])
    return test_score, selected, best_params


def build_oof_base_predictions(X_layers, y_train, base_model_name="logistic_l2", prior_mode="pax"):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    oof = pd.DataFrame(index=y_train.index)

    for omics_name, X_train_raw in X_layers.items():
        scores = pd.Series(index=y_train.index, dtype=float)
        for inner_train_idx, inner_valid_idx in inner_cv.split(X_train_raw, y_train):
            X_inner_train = X_train_raw.iloc[inner_train_idx]
            X_inner_valid = X_train_raw.iloc[inner_valid_idx]
            y_inner_train = y_train.iloc[inner_train_idx]

            selected = select_features_by_omics(
                omics_name,
                X_inner_train,
                prior_mode=prior_mode,
                pax_genes=pax_genes,
                nest_genes=nest_genes,
            )
            model, grid = get_model_spec(base_model_name)
            best_model, _ = tune_model(
                clone(model),
                grid,
                X_inner_train[selected],
                y_inner_train,
            )
            scores.iloc[inner_valid_idx] = positive_class_score(best_model, X_inner_valid[selected])

        oof[omics_name] = scores

    return oof


def evaluate_late_fusion(base_model_name="logistic_l2", prior_mode="pax"):
    X_rna, y = split_x_class_y(rna_paired_df)
    X_cnv, _ = split_x_class_y(cnv_paired_df)
    X_mut, _ = split_x_class_y(mutation_paired_df)

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )

    meta_model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            C=1.0,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_rna, y), start=1):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_layers = {
            "rna_paired": X_rna.iloc[train_idx],
            "cnv": X_cnv.iloc[train_idx],
            "mutation": X_mut.iloc[train_idx],
        }
        test_layers = {
            "rna_paired": X_rna.iloc[test_idx],
            "cnv": X_cnv.iloc[test_idx],
            "mutation": X_mut.iloc[test_idx],
        }

        meta_train = build_oof_base_predictions(
            train_layers,
            y_train,
            base_model_name=base_model_name,
            prior_mode=prior_mode,
        )

        meta_test = pd.DataFrame(index=y_test.index)
        feature_counts = {}
        pax_counts = {}
        nest_counts = {}
        base_params = {}

        for omics_name in ["rna_paired", "cnv", "mutation"]:
            test_score, selected, best_params = fit_predict_one_omics(
                omics_name,
                train_layers[omics_name],
                y_train,
                test_layers[omics_name],
                model_name=base_model_name,
                prior_mode=prior_mode,
            )
            meta_test[omics_name] = test_score
            feature_counts[omics_name] = len(selected)
            pax_counts[omics_name] = len(set(selected) & pax_genes)
            nest_counts[omics_name] = len(set(selected) & nest_genes)
            base_params[omics_name] = best_params

        meta_model.fit(meta_train, y_train)
        y_score = positive_class_score(meta_model, meta_test)
        y_pred = meta_model.predict(meta_test)
        metrics = classification_metrics(y_test, y_pred, y_score)

        rows.append({
            "analysis": "late_fusion_stacking",
            "omics": "rna_cnv_mutation",
            "prior_mode": prior_mode,
            "model": f"stack_{base_model_name}",
            "fold": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_features": meta_train.shape[1],
            "n_rna_features": feature_counts["rna_paired"],
            "n_cnv_features": feature_counts["cnv"],
            "n_mutation_features": feature_counts["mutation"],
            "n_rna_pax_genes": pax_counts["rna_paired"],
            "n_cnv_pax_genes": pax_counts["cnv"],
            "n_mutation_pax_genes": pax_counts["mutation"],
            "base_model": base_model_name,
            "base_params": str(base_params),
            **metrics,
        })

        predictions.extend([
            {
                "analysis": "late_fusion_stacking",
                "omics": "rna_cnv_mutation",
                "prior_mode": prior_mode,
                "model": f"stack_{base_model_name}",
                "fold": fold_id,
                "sample_id": sample_id,
                "y_true": int(true),
                "y_true_label": "Sensitive" if true == 1 else "Resistant",
                "y_pred": int(pred),
                "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                "y_score_sensitive": score,
                "rna_score": meta_test.loc[sample_id, "rna_paired"],
                "cnv_score": meta_test.loc[sample_id, "cnv"],
                "mutation_score": meta_test.loc[sample_id, "mutation"],
            }
            for sample_id, true, pred, score in zip(y_test.index, y_test, y_pred, y_score)
        ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [10]:
print("Running late fusion / stacking...")
late_results_df, late_predictions_df = evaluate_late_fusion(
    base_model_name="logistic_l2",
    prior_mode="pax",
)

late_results_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_results.csv", index=False)
late_predictions_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_predictions.csv", index=False)

late_results_df.groupby(["model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running late fusion / stacking...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
stack_logistic_l2,0.612,0.736,0.56,0.62,0.618,0.501,3.0


In [11]:
RUN_STACKING_SENSITIVITY = True

if RUN_STACKING_SENSITIVITY:

    late_results = []
    late_predictions = []

    for base_model_name in ["logistic_l2", "random_forest"]:
        print(f"Running late fusion / stacking with {base_model_name}...")
        result_df, pred_df = evaluate_late_fusion(
            base_model_name=base_model_name,
            prior_mode="pax",
        )
        late_results.append(result_df)
        late_predictions.append(pred_df)

    late_results_df = pd.concat(late_results, ignore_index=True)
    late_predictions_df = pd.concat(late_predictions, ignore_index=True)

    late_results_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_results.csv", index=False)
    late_predictions_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_predictions.csv", index=False)

    late_results_df.groupby(["model"])[[
        "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
    ]].mean().round(3)

Running late fusion / stacking with logistic_l2...


Running late fusion / stacking with random_forest...


## Compare Model Families

In [12]:
from scipy.stats import ttest_1samp
from scipy.stats import t as t_dist


def fdr_bh(pvals):
    """Benjamini-Hochberg FDR correction."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    return out


all_results_df = pd.concat([single_results_df, early_results_df, late_results_df], ignore_index=True)

# Per model/omics combination: mean AUROC across repeated-CV folds, a 95% t-based
# CI on that mean, and a one-sample t-test of "mean fold AUROC != 0.5 (chance)".
# Caveat: repeated-CV folds are not independent (they reuse the same ~62-80
# samples across repeats), so this t-test is the standard but slightly
# anti-conservative simplification used across most ML papers reporting
# repeated-CV significance, not a variance-corrected test (e.g. Nadeau-Bengio).
# Treat p-values here as indicative, and always read them alongside the CI.
summary_rows = []
for (analysis, omics, model), group in all_results_df.groupby(["analysis", "omics", "model"]):
    aurocs = group["auroc"].to_numpy()
    n_folds = len(aurocs)
    mean = aurocs.mean()
    sd = aurocs.std(ddof=1)
    se = sd / np.sqrt(n_folds)
    tcrit = t_dist.ppf(0.975, df=n_folds - 1)
    t_stat, p_val = ttest_1samp(aurocs, 0.5)

    summary_rows.append({
        "analysis": analysis,
        "omics": omics,
        "model": model,
        "auroc_mean": mean,
        "auroc_sd": sd,
        "auroc_ci_lo": mean - tcrit * se,
        "auroc_ci_hi": mean + tcrit * se,
        "auroc_p_vs_chance": p_val,
        "auprc_mean": group["auprc"].mean(),
        "auprc_sd": group["auprc"].std(),
        "balanced_accuracy_mean": group["balanced_accuracy"].mean(),
        "balanced_accuracy_sd": group["balanced_accuracy"].std(),
        "f1_mean": group["f1"].mean(),
        "sensitivity_mean": group["sensitivity"].mean(),
        "specificity_mean": group["specificity"].mean(),
        "n_features_mean": group["n_features"].mean(),
        "n_folds": n_folds,
    })

summary_df = pd.DataFrame(summary_rows)

# Benjamini-Hochberg FDR correction across all model/omics combinations tested here
summary_df["auroc_p_adj_fdr"] = fdr_bh(summary_df["auroc_p_vs_chance"].to_numpy())

summary_df = summary_df.sort_values(["auroc_mean", "auprc_mean"], ascending=False)

n_sig = (summary_df["auroc_p_adj_fdr"] < 0.05).sum()
print(f"{n_sig}/{len(summary_df)} model/omics combinations significantly better than chance after FDR correction (q < 0.05)")

summary_df.to_csv(RESULTS_DIR / "classification_model_summary.csv", index=False)
summary_df.round(3)

10/12 model/omics combinations significantly better than chance after FDR correction (q < 0.05)


,analysis,omics,model,auroc_mean,auroc_sd,auroc_ci_lo,auroc_ci_hi,auroc_p_vs_chance,auprc_mean,auprc_sd,balanced_accuracy_mean,balanced_accuracy_sd,f1_mean,sensitivity_mean,specificity_mean,n_features_mean,n_folds,auroc_p_adj_fdr
11,single_omics,rna_paired,random_forest,0.716,0.109,0.671,0.761,0.000,0.779,0.106,0.607,0.118,0.744,0.877,0.337,2055.68,25,0.000
1,early_fusion,rna_cnv_mutation,random_forest,0.699,0.101,0.657,0.740,0.000,0.777,0.099,0.601,0.094,0.734,0.861,0.341,2975.36,25,0.000
10,single_omics,rna_paired,logistic_l2,0.671,0.100,0.630,0.712,0.000,0.758,0.091,0.620,0.111,0.667,0.658,0.581,2055.68,25,0.000
3,late_fusion_stacking,rna_cnv_mutation,stack_random_forest,0.658,0.165,0.590,0.726,0.000,0.748,0.121,0.631,0.140,0.632,0.594,0.668,3.00,25,0.000
9,single_omics,rna_all,random_forest,0.657,0.139,0.600,0.714,0.000,0.743,0.114,0.597,0.098,0.733,0.856,0.337,2055.00,25,0.000
8,single_omics,rna_all,logistic_l2,0.618,0.162,0.551,0.685,0.001,0.725,0.125,0.588,0.132,0.620,0.600,0.576,2055.00,25,0.002
2,late_fusion_stacking,rna_cnv_mutation,stack_logistic_l2,0.612,0.120,0.562,0.662,0.000,0.736,0.108,0.560,0.119,0.620,0.618,0.501,3.00,25,0.000
0,early_fusion,rna_cnv_mutation,logistic_l2,0.588,0.135,0.533,0.644,0.003,0.696,0.100,0.610,0.119,0.658,0.659,0.560,2975.36,25,0.004
5,single_omics,cnv,random_forest,0.527,0.136,0.471,0.584,0.327,0.684,0.110,0.508,0.103,0.627,0.691,0.325,492.12,25,0.357
4,single_omics,cnv,logistic_l2,0.510,0.124,0.459,0.561,0.684,0.666,0.094,0.498,0.102,0.556,0.546,0.451,492.12,25,0.684


## Notes for Interpretation

Recommended first-pass readout:

1. Compare `rna_paired` vs `cnv` vs `mutation` to establish single-omics baselines.
2. Compare `early_fusion` against the best single-omics model.
3. If early fusion does not improve, use this as support for late-fusion/stacking as the next model family.
4. Use the saved fold-level predictions for paired statistical comparisons and ROC/PR plots.